# Figure: MAC accuracy vs. opening angle

Loads `results/differentiability/mac_accuracy.json` (produced by `bench/differentiability/mac_accuracy.py`). Never recomputes.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

# Resolve the results directory whether the notebook is run from its own
# folder or from the repo root.
_here = Path.cwd()
_candidates = [
    _here / "results" / "differentiability",
    _here.parents[1] / "results" / "differentiability",
]
RESULTS = next((c for c in _candidates if c.exists()), _candidates[0])
FIGDIR = RESULTS.parents[1] / "examples" / "differentiable_paper" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})
PALETTE = {"radix": "#4C72B0", "octree": "#DD8452", "kdtree": "#55A868"}
BASELINE = "#8C8C8C"


In [ ]:
data = json.loads((RESULTS / "mac_accuracy.json").read_text())
meta = data["metadata"]
records = data["records"]
thetas = [r["theta"] for r in records]
rel_err = [r["rel_error_vs_total"] for r in records]
far_frac = [r["far_energy_fraction"] for r in records]
print(f"N={data['params']['n']} approximation={data['params']['approximation']}")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)

ax.plot(thetas, rel_err, "o-", color=PALETTE["radix"], label="monopole rel. error")
ax.set_xlabel(r"opening angle $\theta$")
ax.set_ylabel(r"$|U_{\rm far}^{\rm mono}-U_{\rm far}^{\rm exact}|\,/\,U_{\rm total}$")
ax.set_yscale("log")

ax2 = ax.twinx()
ax2.plot(thetas, far_frac, "s--", color=PALETTE["octree"], label="far-field energy fraction")
ax2.set_ylabel("fraction of energy handled far", color=PALETTE["octree"])
ax2.tick_params(axis="y", labelcolor=PALETTE["octree"])
ax2.grid(False)

lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], loc="upper left")
ax.set_title(f"Dehnen MAC accuracy vs. cost (N={data['params']['n']})")

out = FIGDIR / "fig_mac_accuracy.pdf"
fig.savefig(out); fig.savefig(out.with_suffix(".png"))
print("wrote", out)
